# Additional scientific analysis of model results

**Scientific question.** Where do LightGBM, SchNet, and their validation-selected ensemble succeed or fail, how uncertain are their measured differences, and which feature families support the classical model?

This notebook reuses the exact splits, saved predictions, metrics, and deterministic feature caches from notebooks 02–04. It does not alter model selection, overwrite checkpoints, or introduce new architectures.

## Notebook contract

**Goal:** provide a technical appendix of robustness and subgroup analyses.  
**Inputs:** saved final predictions, metrics, and deterministic feature metadata.  
**Outputs:** additional diagnostic figures and tables under `outputs/additional_analysis/`.  
**Main questions:** Where do errors concentrate, are model residuals complementary, and how uncertain are the comparisons?


## 1. Setup and artifact validation

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# Standard library
import json

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import sparse

# Project
from src.config import (
    ADDITIONAL_DIR, CLASSICAL_ANALYSIS_FEATURES, CLASSICAL_ANALYSIS_MORGAN,
    CLASSICAL_METRICS, DATA_PATH,
    FINAL_RANDOM_PREDICTIONS, FINAL_SCAFFOLD_PREDICTIONS,
    PROJECT_ROOT, RANDOM_SEED, RESULTS_DIR, SPLIT_ARTIFACT,
)
from src.data import (
    labeled_molecules, load_dataset, load_required_csv, load_required_numpy, require_artifact,
    validate_analysis_features, validate_prediction_alignment, validate_prediction_split,
    validate_split_indices,
)
from src.evaluation import bootstrap_interval, bootstrap_model_comparison, residual_summary
from src.plotting import MODEL_COLORS, configure_matplotlib, save_figure


In [ ]:
ROOT = PROJECT_ROOT
OUTPUT_DIR = ADDITIONAL_DIR
RANDOM_STATE = RANDOM_SEED
rng = np.random.default_rng(RANDOM_STATE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

df = load_dataset(DATA_PATH)
labeled = labeled_molecules(df)
y = labeled["gap"].to_numpy(dtype=np.float32)
random_predictions = load_required_csv(FINAL_RANDOM_PREDICTIONS, "notebooks/04_final_analysis.ipynb")
scaffold_predictions = load_required_csv(FINAL_SCAFFOLD_PREDICTIONS, "notebooks/04_final_analysis.ipynb")
final_metrics = load_required_csv(RESULTS_DIR / "final_model_comparison.csv", "notebooks/04_final_analysis.ipynb")
classical_metrics = load_required_csv(CLASSICAL_METRICS, "notebooks/02_classical_baselines.ipynb")

split_data = load_required_numpy(SPLIT_ARTIFACT, "notebooks/02_classical_baselines.ipynb")
random_test_idx, scaffold_test_idx = split_data["random_test"], split_data["scaffold_test"]
validate_split_indices(split_data["random_train"], split_data["random_validation"], random_test_idx, len(labeled))
validate_split_indices(split_data["scaffold_train"], split_data["scaffold_validation"], scaffold_test_idx, len(labeled))
validate_prediction_alignment(random_predictions, labeled)
validate_prediction_alignment(scaffold_predictions, labeled)
validate_prediction_split(random_predictions, random_test_idx)
validate_prediction_split(scaffold_predictions, scaffold_test_idx)
print("Loaded validated final artifacts; no main-model training is performed.")


## 2. Residual analysis

**Analysis.** Residuals are defined as `prediction − target`. Identical rows are used for all three models within each split. Summary statistics quantify bias, typical error, RMSE sensitivity to extreme targets, and tail behavior.

In [ ]:
MODEL_COLUMNS = {
    "LightGBM": "lightgbm_prediction",
    "SchNet": "schnet_prediction",
    "Ensemble": "ensemble_prediction",
}


residual_table = pd.concat([
    residual_summary(random_predictions, "random", MODEL_COLUMNS),
    residual_summary(scaffold_predictions, "scaffold", MODEL_COLUMNS),
], ignore_index=True)
residual_table.to_csv(OUTPUT_DIR / "residual_summary.csv", index=False)
display(residual_table)


### 2.2 Full-range and robust central residual diagnostics


In [ ]:
for split_name, frame in [("random", random_predictions), ("scaffold", scaffold_predictions)]:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    for row, (model, prediction_column) in enumerate(MODEL_COLUMNS.items()):
        residual = frame[prediction_column] - frame["target"]
        target, prediction = frame["target"], frame[prediction_column]
        full_limits = [min(target.min(), prediction.min()), max(target.max(), prediction.max())]
        axes[row, 0].scatter(target, prediction, s=5, alpha=.20, color=MODEL_COLORS[model], rasterized=True)
        axes[row, 0].plot(full_limits, full_limits, "--", color="black", linewidth=1)
        axes[row, 0].set(xlabel="True target (stored units)", ylabel="Prediction (stored units)",
                         title=f"{model}: full range")

        target_limit = target.quantile(.995)
        residual_limit = residual.abs().quantile(.995)
        mask = (target <= target_limit) & (residual.abs() <= residual_limit)
        axes[row, 1].hexbin(target[mask], residual[mask], gridsize=38, mincnt=1, cmap="viridis")
        axes[row, 1].axhline(0, color="black", linestyle="--", linewidth=1)
        axes[row, 1].set(ylim=(-residual_limit, residual_limit),
                         xlabel="True target (stored units)", ylabel="Residual: prediction − true",
                         title=f"Central 99.5% (n={mask.sum():,})")

        central_residual = residual[residual.abs() <= residual_limit]
        axes[row, 2].hist(central_residual, bins=50, color=MODEL_COLORS[model], edgecolor="white")
        axes[row, 2].axvline(0, color="black", linestyle="--", linewidth=1)
        axes[row, 2].set(xlabel="Residual (stored units)", ylabel="Molecules",
                         title="Central 99.5% residuals")
        axes[row, 2].text(.98, .95,
            f"mean={residual.mean():.3g}\nmedian={residual.median():.3g}\n"
            f"MAE={residual.abs().mean():.3g}\n99.5% |r|={residual_limit:.3g}",
            transform=axes[row, 2].transAxes, ha="right", va="top", fontsize=8)
    fig.suptitle(f"Residual diagnostics — {split_name} test")
    save_figure(f"residual_diagnostics_{split_name}", OUTPUT_DIR)

display(Markdown(
    "Mean residuals diagnose global bias; changes in residual spread with target indicate "
    "heteroscedasticity. RMSE greatly exceeding MAE confirms that the few extreme targets "
    "dominate squared-error metrics, so tail statistics are reported without deleting them."
))

## 3. Error stratified by molecular size

**Analysis.** Atom count, heavy-atom count, bond count, and cached RDKit molecular weight are divided into quartiles. Each bin contains many samples. Bootstrap intervals use 500 fixed-seed resamples within each bin.

In [ ]:
analysis_features = load_required_numpy(
    CLASSICAL_ANALYSIS_FEATURES, "notebooks/02_classical_baselines.ipynb")
validate_analysis_features(analysis_features, labeled)
X_size = analysis_features["X_size"]
X_geometry = analysis_features["X_geometry"]
X_morgan = sparse.load_npz(require_artifact(
    CLASSICAL_ANALYSIS_MORGAN, "notebooks/02_classical_baselines.ipynb")).tocsr()
if X_morgan.shape != (len(labeled), 2048) or X_morgan.data.size and not np.isfinite(X_morgan.data).all():
    raise ValueError(f"Morgan review artifact has invalid shape or values: {X_morgan.shape}")
for frame in [random_predictions, scaffold_predictions]:
    indices = frame["dataset_index"].to_numpy(dtype=int)
    frame["molecular_weight"] = analysis_features["molecular_weight"][indices]


size_rows = []
size_features = ["n_atoms", "n_heavy_atoms", "n_bonds", "molecular_weight"]
for split_name, frame in [("random", random_predictions), ("scaffold", scaffold_predictions)]:
    for feature in size_features:
        bins = pd.qcut(frame[feature], q=4, duplicates="drop")
        for bin_label, group in frame.assign(size_bin=bins).groupby("size_bin", observed=True):
            for model, prediction_column in MODEL_COLUMNS.items():
                absolute_error = (group[prediction_column] - group["target"]).abs().to_numpy()
                ci_low, ci_high = bootstrap_interval(absolute_error, repetitions=500, seed=RANDOM_STATE + len(size_rows))
                size_rows.append({"split": split_name, "size_feature": feature, "bin": str(bin_label),
                    "bin_midpoint": float(bin_label.mid), "model": model, "samples": len(group),
                    "mae": absolute_error.mean(), "mae_ci_low": ci_low, "mae_ci_high": ci_high})
size_table = pd.DataFrame(size_rows)
size_table.to_csv(OUTPUT_DIR / "size_stratified_errors.csv", index=False)
display(size_table)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, size_features):
    subset = size_table[(size_table["split"] == "random") & (size_table["size_feature"] == feature)]
    labels = (subset[subset["model"] == "LightGBM"].sort_values("bin_midpoint")["bin"].tolist())
    positions = np.arange(len(labels))
    for model in MODEL_COLUMNS:
        values = subset[subset["model"] == model].sort_values("bin_midpoint")
        ax.errorbar(positions, values["mae"],
                    yerr=[values["mae"] - values["mae_ci_low"], values["mae_ci_high"] - values["mae"]],
                    marker="o", capsize=3, color=MODEL_COLORS[model], label=model)
    counts = subset[subset["model"] == "LightGBM"].sort_values("bin_midpoint")["samples"].to_numpy()
    ax.set_xticks(positions, [f"{label}\n(n={count})" for label, count in zip(labels, counts)],
                  rotation=20, ha="right")
    ax.set(xlabel=feature.replace("_", " "), ylabel="MAE (stored units)",
           title=feature.replace("_", " ").title())
axes[0, 0].legend(frameon=False)
fig.suptitle("Random-test error by molecular-size quartile (bootstrap 95% CI)")
save_figure("error_by_molecular_size", OUTPUT_DIR)
display(Markdown("Quartiles are treated as categories, use identical bins for all models, and show their sample counts; lines connect ordered size groups only."))

size_gain = (
    size_table.query("split == 'random'").pivot_table(index=["size_feature", "bin_midpoint"], columns="model", values="mae")
)
size_gain["schnet_gain_over_lightgbm"] = size_gain["LightGBM"] - size_gain["SchNet"]
display(Markdown("**Geometry-aware gain by size bin (positive favors SchNet)**"))
display(size_gain)

## 4. Error versus chemical novelty

**Scientific value.** Scaffold test molecules have unseen scaffolds by construction, so scaffold frequency is always zero. To obtain a graded novelty measure without an expensive full all-pairs matrix, maximum Morgan Tanimoto similarity is approximated against a fixed 5,000-molecule subset of scaffold-training data using sparse batched products.

In [ ]:
scaffolds_all = analysis_features["scaffolds"]
scaffold_groups = {}
for idx, scaffold in enumerate(scaffolds_all):
    scaffold_groups.setdefault(scaffold, []).append(idx)
ordered_groups = sorted(scaffold_groups.items(), key=lambda item: (-len(item[1]), item[0]))
train_cutoff, validation_cutoff = int(.70 * len(labeled)), int(.85 * len(labeled))
scaffold_train_exact, scaffold_val_exact, scaffold_test_exact = [], [], []
for _, group_indices in ordered_groups:
    if len(scaffold_train_exact) + len(group_indices) <= train_cutoff:
        scaffold_train_exact.extend(group_indices)
    elif len(scaffold_train_exact) + len(scaffold_val_exact) + len(group_indices) <= validation_cutoff:
        scaffold_val_exact.extend(group_indices)
    else:
        scaffold_test_exact.extend(group_indices)
scaffold_train_idx = np.asarray(scaffold_train_exact, dtype=int)
assert np.array_equal(np.asarray(scaffold_test_exact), scaffold_predictions["dataset_index"].to_numpy())
reference_rng = np.random.default_rng(RANDOM_STATE)
reference_idx = np.sort(reference_rng.choice(scaffold_train_idx, size=min(5000, len(scaffold_train_idx)), replace=False))
test_idx = scaffold_predictions["dataset_index"].to_numpy(dtype=int)
reference_fp, test_fp = X_morgan[reference_idx].astype(np.float32), X_morgan[test_idx].astype(np.float32)
reference_active = np.asarray(reference_fp.sum(axis=1)).ravel()
test_active = np.asarray(test_fp.sum(axis=1)).ravel()
max_similarity = np.empty(len(test_idx), dtype=np.float32)
for start in range(0, len(test_idx), 256):
    stop = min(start + 256, len(test_idx))
    intersection = (test_fp[start:stop] @ reference_fp.T).toarray()
    union = test_active[start:stop, None] + reference_active[None, :] - intersection
    max_similarity[start:stop] = np.max(intersection / np.maximum(union, 1), axis=1)
scaffold_predictions["approx_max_train_tanimoto"] = max_similarity
similarity_bins = pd.qcut(scaffold_predictions["approx_max_train_tanimoto"], q=4, duplicates="drop")
novelty_rows = []
for bin_label, group in scaffold_predictions.assign(similarity_bin=similarity_bins).groupby("similarity_bin", observed=True):
    for model, prediction_column in MODEL_COLUMNS.items():
        error = (group[prediction_column] - group["target"]).abs()
        novelty_rows.append({"similarity_bin": str(bin_label), "similarity_midpoint": float(bin_label.mid),
                             "model": model, "samples": len(group), "mae": error.mean()})
novelty_table = pd.DataFrame(novelty_rows)
novelty_table.to_csv(OUTPUT_DIR / "novelty_stratified_errors.csv", index=False)
display(novelty_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
ordered = novelty_table[novelty_table["model"] == "LightGBM"].sort_values("similarity_midpoint", ascending=False)
labels = ordered["similarity_bin"].tolist()
positions = np.arange(len(labels))
for model in MODEL_COLUMNS:
    values = novelty_table[novelty_table["model"] == model].set_index("similarity_bin").loc[labels]
    ax.scatter(positions, values["mae"], s=48, color=MODEL_COLORS[model], label=model, zorder=3)
counts = ordered["samples"].to_numpy()
ax.set_xticks(positions, [f"{label}\n(n={count})" for label, count in zip(labels, counts)])
ax.set_xlabel("Maximum train Tanimoto bin (most familiar → most novel)")
ax.set_ylabel("Scaffold-test MAE (stored units)")
ax.set_title("Error versus chemical novelty")
ax.legend(frameon=False)
save_figure("error_vs_chemical_novelty", OUTPUT_DIR)
display(Markdown("Similarity quartiles are categorical groups ordered from most familiar to most novel; no connecting line implies unsupported continuity. Counts are shown for every shared bin."))

## 5. Fixed-configuration feature-family ablation

**Analysis.** The selected LightGBM estimator configuration is held fixed. No feature-specific search is performed. Five families are fitted on the same random training subset and evaluated on unchanged validation/test subsets.

In [ ]:
ablation_table = load_required_csv(
    OUTPUT_DIR / "feature_family_ablation.csv", "notebooks/02_classical_baselines.ipynb")
display(ablation_table)
display(Markdown(
    "This appendix loads the fixed-model ablation artifact produced during the validated analysis; "
    "it does not refit models."
))

plot_ablation = ablation_table.sort_values("validation_mae", ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
ypos = np.arange(len(plot_ablation))
ax.barh(ypos - .18, plot_ablation["validation_mae"], height=.36, color="#4C78A8", label="Validation")
ax.barh(ypos + .18, plot_ablation["test_mae"], height=.36, color="#F58518", label="Test")
ax.set_yticks(ypos, plot_ablation["feature_family"])
ax.set_xlim(left=0)
ax.set_xlabel("MAE (stored units)")
ax.set_title("Fixed-LightGBM feature-family ablation")
ax.legend(frameon=False)
for y0, value in zip(ypos - .18, plot_ablation["validation_mae"]):
    ax.text(value, y0, f" {value:.4f}", va="center", fontsize=8)
for y0, value in zip(ypos + .18, plot_ablation["test_mae"]):
    ax.text(value, y0, f" {value:.4f}", va="center", fontsize=8)
save_figure("feature_family_ablation", OUTPUT_DIR)
display(Markdown("The linear axis begins at zero, and horizontal grouping keeps the long feature-family labels readable without exaggerating small MAE differences."))

## 6. Model-error complementarity

**Analysis.** Signed residual correlations measure shared error direction; absolute-error correlations measure shared difficulty. Positive error difference means SchNet is more accurate for that molecule.

In [ ]:
complementarity_rows = []
for split_name, frame in [("random", random_predictions), ("scaffold", scaffold_predictions)]:
    classical_residual = frame["lightgbm_prediction"] - frame["target"]
    schnet_residual = frame["schnet_prediction"] - frame["target"]
    classical_abs, schnet_abs = classical_residual.abs(), schnet_residual.abs()
    ties = np.isclose(classical_abs, schnet_abs, atol=1e-12)
    complementarity_rows.append({"split": split_name,
        "signed_residual_pearson": classical_residual.corr(schnet_residual, method="pearson"),
        "signed_residual_spearman": classical_residual.corr(schnet_residual, method="spearman"),
        "absolute_error_pearson": classical_abs.corr(schnet_abs, method="pearson"),
        "absolute_error_spearman": classical_abs.corr(schnet_abs, method="spearman"),
        "lightgbm_lower_error_proportion": (classical_abs < schnet_abs).mean(),
        "schnet_lower_error_proportion": (schnet_abs < classical_abs).mean(),
        "tie_proportion": ties.mean()})
complementarity_table = pd.DataFrame(complementarity_rows)
complementarity_table.to_csv(OUTPUT_DIR / "model_error_complementarity.csv", index=False)
display(complementarity_table)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, (split_name, frame) in zip(
    axes, [("random", random_predictions), ("scaffold", scaffold_predictions)]
):
    classical_residual = frame["lightgbm_prediction"] - frame["target"]
    schnet_residual = frame["schnet_prediction"] - frame["target"]
    limit = np.quantile(np.abs(np.r_[classical_residual, schnet_residual]), .995)
    mask = (classical_residual.abs() <= limit) & (schnet_residual.abs() <= limit)
    ax.hexbin(classical_residual[mask], schnet_residual[mask], gridsize=42, mincnt=1, cmap="viridis")
    ax.axhline(0, color="black", linewidth=.8); ax.axvline(0, color="black", linewidth=.8)
    ax.plot([-limit, limit], [-limit, limit], "--", color="white", linewidth=1)
    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), aspect="equal",
           xlabel="LightGBM residual (stored units)", ylabel="SchNet residual (stored units)",
           title=f"{split_name.title()} test: central 99.5%")
    row = complementarity_table.set_index("split").loc[split_name]
    ax.text(.03, .97, f"Pearson={row.signed_residual_pearson:.2f}\nSpearman={row.signed_residual_spearman:.2f}\n"
            f"LightGBM wins={row.lightgbm_lower_error_proportion:.1%}\nSchNet wins={row.schnet_lower_error_proportion:.1%}",
            transform=ax.transAxes, va="top", color="white",
            bbox={"facecolor": "black", "alpha": .55, "edgecolor": "none"})
fig.suptitle("Signed residual complementarity (density)")
save_figure("residual_complementarity", OUTPUT_DIR)
display(Markdown("Hexagonal density reveals the central residual structure without logarithmic axes. The symmetric 99.5% visual limit excludes only extreme points from display; correlations and win proportions use all molecules."))


## 7. Bootstrap confidence intervals and paired effects

**Analysis.** One thousand paired bootstrap samples preserve molecule-level correspondence. Confidence intervals emphasize effect size and uncertainty rather than p-values.

In [ ]:

bootstrap_metrics_parts, paired_parts = [], []
for offset, (split_name, frame) in enumerate([
    ("random", random_predictions), ("scaffold", scaffold_predictions)
]):
    metrics_part, paired_part = bootstrap_model_comparison(
        frame, split_name, MODEL_COLUMNS, repetitions=1000, seed=RANDOM_STATE + offset
    )
    bootstrap_metrics_parts.append(metrics_part)
    paired_parts.append(paired_part)
bootstrap_metrics = pd.concat(bootstrap_metrics_parts, ignore_index=True)
paired_bootstrap = pd.concat(paired_parts, ignore_index=True)
bootstrap_metrics.to_csv(OUTPUT_DIR / "bootstrap_metric_intervals.csv", index=False)
paired_bootstrap.to_csv(OUTPUT_DIR / "bootstrap_paired_differences.csv", index=False)
display(bootstrap_metrics)
display(paired_bootstrap)

mae_intervals = bootstrap_metrics.query("metric == 'MAE'")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
for ax, split_name in zip(axes, ["random", "scaffold"]):
    values = mae_intervals[mae_intervals["split"] == split_name]
    ax.errorbar(
        values["model"], values["estimate"],
        yerr=[values["estimate"] - values["ci_low"],
              values["ci_high"] - values["estimate"]],
        fmt="o", capsize=4,
    )
    ax.set(title=f"{split_name.title()} test", ylabel="MAE", xlabel="")
fig.suptitle("Bootstrap 95% confidence intervals")
save_figure("bootstrap_mae_intervals", OUTPUT_DIR)

## 8. Worst-case analysis

**Analysis.** The final ensemble is the best model by held-out MAE. The 15 largest ensemble errors are reported with structural metadata and approximate training similarity. Patterns are observations and hypotheses, not mechanistic chemical claims.

In [ ]:
worst_source = scaffold_predictions.copy()
worst_source["ensemble_residual"] = worst_source["ensemble_prediction"] - worst_source["target"]
worst_source["ensemble_absolute_error"] = worst_source["ensemble_residual"].abs()
worst_source["approx_max_train_tanimoto"] = max_similarity
element_columns = ["element_C", "element_N", "element_O", "element_F"]
worst_source["elements_present"] = worst_source.apply(
    lambda row: ",".join(
        column.replace("element_", "") for column in element_columns if row[column] > 0
    ), axis=1
)
worst_columns = [
    "mol_id", "target", "ensemble_prediction", "ensemble_residual",
    "ensemble_absolute_error", "n_atoms", "n_heavy_atoms", "n_bonds",
    "scaffold", "scaffold_train_frequency", "approx_max_train_tanimoto",
    "elements_present", "smiles",
]
worst_table = worst_source.nlargest(15, "ensemble_absolute_error")[worst_columns]
worst_table.to_csv(OUTPUT_DIR / "worst_case_molecules.csv", index=False)
display(worst_table)

worst_summary = pd.Series({
    "median_atoms_worst": worst_table["n_atoms"].median(),
    "median_atoms_all_scaffold_test": scaffold_predictions["n_atoms"].median(),
    "median_similarity_worst": worst_table["approx_max_train_tanimoto"].median(),
    "median_similarity_all_scaffold_test": np.median(max_similarity),
    "fluorine_fraction_worst": worst_table["elements_present"].str.contains("F").mean(),
    "fluorine_fraction_all": (scaffold_predictions["element_F"] > 0).mean(),
})
display(worst_summary.to_frame("value"))
display(Markdown(
    "Recurring size, novelty, or element patterns are descriptive signals for follow-up. "
    "They may reflect sparse coverage, target outliers, or model limitations; the table alone "
    "does not support causal chemical conclusions."
))

## 9. Optional dataset-size learning curve

Skipped intentionally. Although the classical training function is reusable, five fractions × three deterministic seeds would require 15 additional complete LightGBM fits. This is disproportionate to the requested interpretation-focused analysis and is not needed to support the core residual, novelty, ablation, complementarity, and uncertainty conclusions.

## 10. Consolidated results

In [ ]:
consolidated_results = pd.concat([
    residual_table.assign(analysis="residual_metrics"),
    ablation_table.rename(columns={"feature_family": "model"}).assign(
        split="random", analysis="feature_ablation"
    ),
    novelty_table.assign(split="scaffold", analysis="chemical_novelty"),
    complementarity_table.assign(analysis="error_complementarity"),
    paired_bootstrap.assign(analysis="paired_bootstrap"),
], ignore_index=True, sort=False)
consolidated_results.to_csv(OUTPUT_DIR / "consolidated_results.csv", index=False)
display(consolidated_results)

## Main scientific conclusions

In [ ]:
random_paired = paired_bootstrap.query("split == 'random'").set_index("comparison")
scaffold_paired = paired_bootstrap.query("split == 'scaffold'").set_index("comparison")
largest_size_gain = size_gain["schnet_gain_over_lightgbm"].idxmax()
novelty_pivot = novelty_table.pivot(index="similarity_midpoint", columns="model", values="mae")
novelty_pivot["schnet_gain"] = novelty_pivot["LightGBM"] - novelty_pivot["SchNet"]
most_novel_gain = novelty_pivot.sort_index().iloc[0]["schnet_gain"]
best_ablation = ablation_table.iloc[0]

display(Markdown(f'''
- The ensemble has the lowest MAE on both saved test sets; its paired bootstrap advantage over the best individual model is {random_paired.loc["Best individual minus Ensemble", "mean_paired_difference"]:.5f} on random test and {scaffold_paired.loc["Best individual minus Ensemble", "mean_paired_difference"]:.5f} on scaffold test.
- SchNet's strongest size-stratified gain occurs for `{largest_size_gain}`, while the most novel scaffold-test quartile changes MAE by {most_novel_gain:+.5f} in SchNet's favor.
- Fixed-configuration ablation identifies `{best_ablation["feature_family"]}` as the strongest classical feature family/set (validation MAE {best_ablation["validation_mae"]:.5f}); this measures feature contribution without family-specific tuning.
- Residual rank correlations and the per-molecule win proportions show meaningful but incomplete complementarity, supporting the validation-selected ensemble.
- Extreme targets dominate RMSE and Pearson residual correlation. MAE, median error, tail quantiles, and paired intervals provide a more representative assessment.
'''))

## Limitations

- The ensemble was proposed after earlier test results had been inspected; its weight is validation-only, but the ensemble finding remains exploratory until confirmed on a fresh holdout.
- Stored energy units appear inconsistent with the README and should be confirmed with Aqemia.
- Scaffold-test similarity is approximate, using a fixed 5,000-molecule training reference rather than all training molecules.
- Confidence intervals quantify sampling uncertainty for these fixed test sets; they do not cover split-selection, model-selection, or dataset-shift uncertainty.
- Worst-molecule patterns are descriptive and should not be interpreted as causal chemistry.

## Slide-ready takeaways

- The validation-weighted ensemble is best on random and scaffold tests.
- SchNet is slightly weaker for interpolation but stronger on unseen scaffolds.
- Model errors are complementary enough for a measurable ensemble gain.
- Full 2D + 3D classical features outperform isolated feature families.
- Extreme targets dominate squared-error metrics; paired MAE uncertainty is the clearest comparison.

## Saved artifacts and interpretation

The notebook's scientific findings and conclusions are stated in the preceding sections. Its declared outputs are saved at the paths listed in the **Notebook contract** above. Implementation details shared across experiments live in `src/`; experiment choices and their interpretation remain visible here.
